# Cuaderno B · Modelo de IA de sensado virtual de PM10

**LlajtaLab · NASA Space Apps 2026 Cochabamba · Enfoque 1**

La IA aprende cómo se relaciona lo que ve el satélite con el PM10 que miden las 2 estaciones de la Red MÓNICA, y con eso **estima el PM10 en los 15 distritos**. Es el requisito central del Enfoque 1.

| Decisión | Qué usamos |
|---|---|
| Qué predice | PM10 promedio diario (µg/m³) |
| Variables de entrada | Aerosoles (AOD), calor del suelo, temperatura, viento, humedad y capa de mezcla (MERRA-2), lluvia (GPM), **focos de incendio de la región (FIRMS)**, mes y día de la semana |
| Modelo | Gradient Boosting (`HistGradientBoostingRegressor`), acepta días sin dato satelital |
| Validación 1 | Entrenar con jul. 2022 – dic. 2024 y evaluar con 2025–2026 |
| Validación 2 | Entrenar con una estación y predecir la otra (la prueba honesta del sensado virtual) |
| Comparación | Línea base: el promedio de cada mes |
| Incertidumbre | Rango probable del 10 % al 90 % |

**Sube a este Colab:**
1. `distritos_indicadores_ambientales_2024.geojson`
2. Los dos CSV de la Red MÓNICA (*kanata* y *semapa*)
3. `red_monica_cochabamba.json` (está en `public/data/`): de ahí se sacan las coordenadas de las estaciones
4. *(Opcional)* `variables_diarias.csv`, si ya corriste la celda B2 antes y Colab se cortó: así no la repites.


## Celda B1 · Preparación y coordenadas de las estaciones

In [ ]:
!pip install -q earthengine-api pymannkendall
import ee, json, time, glob, os, pandas as pd, numpy as np, pymannkendall as mk
from scipy import stats
ee.Authenticate(); ee.Initialize(project='microclimas-cochabamba')

gj = json.load(open('/content/distritos_indicadores_ambientales_2024.geojson', encoding='utf-8'))
CAMPO_ID, CAMPO_NOMBRE = 'codigo_distrito', 'nombre'
cod = lambda v: int(float(v))
NOMBRES = {f"d_{cod(f['properties'][CAMPO_ID])}": f['properties'][CAMPO_NOMBRE] for f in gj['features']}
INICIO, FIN = '2022-07-01', '2026-10-01'

def buscar_csv(palabra):
    cand = [f for f in glob.glob('/content/*.csv') if palabra in f.lower()]
    if not cand:
        raise FileNotFoundError(f'No encuentro el CSV de {palabra}: súbelo a /content')
    return cand[0]
RUTAS = {'Kanata': buscar_csv('kanata'), 'SEMAPA': buscar_csv('semapa')}

# Coordenadas (longitud, latitud) tomadas de red_monica_cochabamba.json
ESTACIONES = {'Kanata': (-66.1554, -17.4132), 'SEMAPA': (-66.1602, -17.3685)}

# Zonas: 15 distritos + un círculo de 1,5 km alrededor de cada estación
feats = [{'type': 'Feature', 'geometry': f['geometry'], 'properties': {'zona': f"d_{cod(f['properties'][CAMPO_ID])}"}}
         for f in gj['features']]
zonas = ee.FeatureCollection({'type': 'FeatureCollection', 'features': feats}).merge(
    ee.FeatureCollection([ee.Feature(ee.Geometry.Point(list(xy)).buffer(1500), {'zona': f'est_{n}'})
                          for n, xy in ESTACIONES.items()]))
municipio = ee.FeatureCollection(gj).geometry().dissolve(100)
print('Listo:', len(feats), 'distritos + 2 estaciones')
print('CSV encontrados:', RUTAS)

Listo: 15 distritos + 2 estaciones
CSV encontrados: {'Kanata': '/content/red_monica.kanata (1).csv', 'SEMAPA': '/content/red_monica.semapa (1).csv'}


## Celda B2 · Variables satelitales diarias (la más lenta)

Una consulta por mes, con reintentos. Son unos 52 meses. Si ya tienes `variables_diarias.csv` de una corrida anterior subido a `/content`, esta celda lo usa y no repite nada.

**Al terminar, descarga `variables_diarias.csv`** (respaldo).

In [ ]:
aod_c = ee.ImageCollection('MODIS/061/MCD19A2_GRANULES').select('Optical_Depth_055')
lst_c = ee.ImageCollection('MODIS/061/MOD11A1').select('LST_Day_1km')
slv_c = ee.ImageCollection('NASA/GSFC/MERRA/slv/2')
flx_c = ee.ImageCollection('NASA/GSFC/MERRA/flx/2').select('PBLH')
imerg_c = ee.ImageCollection('NASA/GPM_L3/IMERG_V07').select('precipitation')
vacia = ee.Image.constant(0).updateMask(ee.Image.constant(0))
ARCH = '/content/variables_diarias.csv'
VAR_N = ['aod', 'lst', 't2m', 'u10', 'v10', 'humedad', 'capa_mezcla', 'lluvia']

def m(col, a, b):
    c = col.filterDate(a, b)
    return ee.Image(ee.Algorithms.If(c.size().gt(0), c.mean(), vacia))

def img_dias(dias):
    bandas = []
    for d in dias:
        a = ee.Date(d.strftime('%Y-%m-%d')); b = a.advance(1, 'day'); s = d.strftime('%Y%m%d')
        bandas += [m(aod_c, a, b).multiply(0.001).rename(f'aod_{s}'),
                   m(lst_c, a, b).multiply(0.02).subtract(273.15).rename(f'lst_{s}'),
                   m(slv_c.select('T2M'), a, b).subtract(273.15).rename(f't2m_{s}'),
                   m(slv_c.select('U10M'), a, b).rename(f'u10_{s}'),
                   m(slv_c.select('V10M'), a, b).rename(f'v10_{s}'),
                   m(slv_c.select('QV2M'), a, b).multiply(1000).rename(f'humedad_{s}'),
                   m(flx_c, a, b).rename(f'capa_mezcla_{s}'),
                   m(imerg_c, a, b).multiply(24).rename(f'lluvia_{s}')]
    return ee.Image.cat(bandas)

def bajar(dias):
    res = (img_dias(dias).reduceRegions(collection=zonas, reducer=ee.Reducer.mean(), scale=1000, tileScale=8)
           .map(lambda f: f.setGeometry(None)).getInfo()['features'])
    out = []
    for f in res:
        p = f['properties']
        for d in dias:
            s = d.strftime('%Y%m%d')
            out.append({'zona': p['zona'], 'fecha': d, **{v: p.get(f'{v}_{s}') for v in VAR_N}})
    return out

hechos = set()
if os.path.exists(ARCH):
    previo = pd.read_csv(ARCH, parse_dates=['fecha'])
    hechos = set(previo['fecha'].dt.strftime('%Y-%m'))
    print('Meses ya hechos:', len(hechos))

for ini in pd.date_range(INICIO, FIN, freq='MS'):
    if ini.strftime('%Y-%m') in hechos:
        continue
    dias = list(pd.date_range(ini, ini + pd.offsets.MonthEnd(0)))
    t0 = time.time(); filas_mes = None
    for intento in range(4):
        try:
            if intento < 2:
                filas_mes = bajar(dias)                                   # el mes completo
            else:
                filas_mes = bajar(dias[:15]) + bajar(dias[15:])           # si falla, en dos mitades
            break
        except Exception as e:
            print(ini.strftime('%Y-%m'), 'reintento:', str(e)[:70]); time.sleep(30 * (intento + 1))
    if filas_mes is None:
        print('⚠ No se pudo', ini.strftime('%Y-%m'), '— se sigue con el siguiente'); continue
    pd.DataFrame(filas_mes).to_csv(ARCH, mode='a', header=not os.path.exists(ARCH), index=False)
    print('Listo', ini.strftime('%Y-%m'), f'({time.time() - t0:.0f} s)')

X = pd.read_csv(ARCH, parse_dates=['fecha']).drop_duplicates(subset=['zona', 'fecha'])
print(X.shape); print(X.describe().round(2))

Listo 2022-07 (5 s)
Listo 2022-08 (2 s)
Listo 2022-09 (2 s)
Listo 2022-10 (2 s)
Listo 2022-11 (7 s)
Listo 2022-12 (13 s)
Listo 2023-01 (5 s)
Listo 2023-02 (2 s)
Listo 2023-03 (4 s)
Listo 2023-04 (3 s)
Listo 2023-05 (14 s)
Listo 2023-06 (2 s)
Listo 2023-07 (3 s)
Listo 2023-08 (2 s)
Listo 2023-09 (10 s)
Listo 2023-10 (2 s)
Listo 2023-11 (3 s)
Listo 2023-12 (2 s)
Listo 2024-01 (2 s)
Listo 2024-02 (4 s)
Listo 2024-03 (3 s)
Listo 2024-04 (2 s)
Listo 2024-05 (6 s)
Listo 2024-06 (2 s)
Listo 2024-07 (3 s)
Listo 2024-08 (2 s)
Listo 2024-09 (2 s)
Listo 2024-10 (3 s)
Listo 2024-11 (3 s)
Listo 2024-12 (2 s)
Listo 2025-01 (4 s)
Listo 2025-02 (4 s)
Listo 2025-03 (7 s)
Listo 2025-04 (2 s)
Listo 2025-05 (3 s)
Listo 2025-06 (3 s)
Listo 2025-07 (3 s)
Listo 2025-08 (4 s)
Listo 2025-09 (3 s)
Listo 2025-10 (12 s)
Listo 2025-11 (2 s)
Listo 2025-12 (2 s)
Listo 2026-01 (4 s)
Listo 2026-02 (2 s)
Listo 2026-03 (3 s)
Listo 2026-04 (2 s)
Listo 2026-05 (4 s)
Listo 2026-06 (7 s)
Listo 2026-07 (3 s)
Listo 2026-08 (2

## Celda B2b · Focos de incendio diarios de la región (FIRMS)

En el Cuaderno A vimos que los meses con más incendios tienen más PM10. Por eso le damos a la IA el número de focos de cada día en 150 km alrededor de la ciudad. Si esta celda falla, no pasa nada: el modelo sigue sin esta variable.

In [ ]:
firms = ee.ImageCollection('FIRMS').select('T21')
region150 = municipio.buffer(150000)

def focos_dias(ini):
    dias = pd.date_range(ini, ini + pd.offsets.MonthEnd(0))
    lista = ee.List([firms.filterDate(d.strftime('%Y-%m-%d'), (d + pd.Timedelta(days=1)).strftime('%Y-%m-%d'))
                     .map(lambda i: i.gt(0)).sum().reduceRegion(ee.Reducer.sum(), region150, 1000, maxPixels=1e10).get('T21')
                     for d in dias])
    return dias, lista

try:
    filas = []
    for ini in pd.date_range(INICIO, FIN, freq='MS'):
        dias, lista = focos_dias(ini)
        for intento in range(6):
            try:
                vals = lista.getInfo(); break
            except Exception as e:
                print(ini.strftime('%Y-%m'), 'reintento'); time.sleep(20 * (intento + 1))
        else:
            vals = [None] * len(dias)
        filas += [{'fecha': d, 'focos': (v or 0)} for d, v in zip(dias, vals)]
    FD = pd.DataFrame(filas)
    X = X.drop(columns=['focos'], errors='ignore').merge(FD, on='fecha', how='left')
    print('Focos agregados. Días con algún foco:', int((FD['focos'] > 0).sum()), 'de', len(FD))
except Exception as e:
    print('No se pudieron agregar los focos, se sigue sin esa variable:', str(e)[:120])

Focos agregados. Días con algún foco: 638 de 1584


## Celda B3 · Unir con el PM10 real

In [11]:
X = X.sort_values(['zona', 'fecha']).reset_index(drop=True)
g = X.groupby('zona')
X['viento'] = np.hypot(X['u10'], X['v10'])
X['lluvia_3d'] = g['lluvia'].transform(lambda s: s.rolling(3, min_periods=1).sum())
X['aod_7d'] = g['aod'].transform(lambda s: s.rolling(7, min_periods=1).mean())
X['dias_secos'] = g['lluvia'].transform(lambda s: (s < 1).astype(int).groupby((s >= 1).cumsum()).cumsum())
doy = X['fecha'].dt.dayofyear
X['estacion_sen'] = np.sin(2 * np.pi * doy / 365.25); X['estacion_cos'] = np.cos(2 * np.pi * doy / 365.25)
X['mes'] = X['fecha'].dt.month; X['dsem'] = X['fecha'].dt.dayofweek
if 'focos' in X.columns:
    X['focos_7d'] = g['focos'].transform(lambda s: s.rolling(7, min_periods=1).sum())

def pm10_diario(ruta):
    d = pd.read_csv(ruta, parse_dates=['fecha_medicion'])
    d = d[(d['pm10'] > 0) & (d['pm10'] < 1000)].set_index('fecha_medicion')['pm10']
    dia = d.resample('D').agg(['mean', 'count'])
    return dia[dia['count'] >= 18]['mean']

pm = pd.concat([pm10_diario(r).rename('pm10').rename_axis('fecha').reset_index().assign(zona=f'est_{n}')
                for n, r in RUTAS.items()])
D = X.merge(pm, on=['zona', 'fecha'], how='inner').reset_index(drop=True)

VARS_FISICAS = [v for v in ['aod', 'aod_7d', 'lst', 't2m', 'viento', 'u10', 'v10', 'humedad', 'capa_mezcla',
                            'lluvia', 'lluvia_3d', 'dias_secos', 'focos', 'focos_7d'] if v in X.columns]
VARS = VARS_FISICAS + ['estacion_sen', 'estacion_cos', 'dsem']
print('Variables del modelo:', VARS)
print('Días para entrenar:', D.groupby('zona').size().to_dict())

Variables del modelo: ['aod', 'aod_7d', 'lst', 't2m', 'viento', 'u10', 'v10', 'humedad', 'capa_mezcla', 'lluvia', 'lluvia_3d', 'dias_secos', 'focos', 'focos_7d', 'estacion_sen', 'estacion_cos', 'dsem']
Días para entrenar: {'est_Kanata': 1097, 'est_SEMAPA': 1467}


## Celda B4 · Validación (las métricas para el jurado)

**Cómo leerlo:** si el R² del modelo es mayor que el de la línea base, la IA aporta. Si "Kanata predice SEMAPA" sale bajo, **no se esconde**: se muestra y se explica que con 2 estaciones la estimación en otros distritos es aproximada.

In [12]:
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import GroupKFold
from sklearn.inspection import permutation_importance

def modelo(q=None):
    kw = dict(max_iter=400, learning_rate=0.04, max_leaf_nodes=15, min_samples_leaf=30, l2_regularization=1.0, random_state=0)
    return HGB(loss='quantile', quantile=q, **kw) if q else HGB(**kw)
def entrenar(tr, q=None):
    return modelo(q).fit(tr[VARS], np.log(tr['pm10']))     # en logaritmo: el PM10 tiene picos muy altos
def predecir(m, df):
    return np.exp(m.predict(df[VARS]))
def met(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return {'r2': round(float(r2_score(y, p)), 3), 'rmse': round(float(mean_squared_error(y, p) ** 0.5), 1),
            'mae': round(float(mean_absolute_error(y, p)), 1), 'correlacion': round(float(np.corrcoef(y, p)[0, 1]), 2), 'n': int(len(y))}
def base(tr, te):
    return te['mes'].map(tr.groupby('mes')['pm10'].mean()).fillna(tr['pm10'].mean()).values

val = {}
# 1) Meses que el modelo nunca vio (validación principal)
grupos = D['fecha'].dt.to_period('M').astype(str)
D['estimado'] = np.nan; D['base'] = np.nan
for tri, tei in GroupKFold(5).split(D, groups=grupos):
    tr, te = D.iloc[tri], D.iloc[tei]
    m_cv = entrenar(tr)
    D.loc[te.index, 'estimado'] = predecir(m_cv, te)
    D.loc[te.index, 'base'] = base(tr, te)
val['dias_no_vistos'] = {'modelo': met(D['pm10'], D['estimado']), 'linea_base': met(D['pm10'], D['base'])}
MM = D.groupby(['zona', D['fecha'].dt.to_period('M')])[['pm10', 'estimado', 'base']].mean().dropna()
val['promedios_mensuales'] = {'modelo': met(MM['pm10'], MM['estimado']), 'linea_base': met(MM['pm10'], MM['base'])}

# 2) ¿Predice el futuro?
tr, te = D[D['fecha'] < '2025-01-01'], D[D['fecha'] >= '2025-01-01']
val['futuro_2025_2026'] = {'modelo': met(te['pm10'], predecir(entrenar(tr), te)), 'linea_base': met(te['pm10'], base(tr, te))}

# 3) ¿Sirve donde no hay sensor?
for a, b in [('est_Kanata', 'est_SEMAPA'), ('est_SEMAPA', 'est_Kanata')]:
    tr, te = D[D['zona'] == a], D[D['zona'] == b]
    val[f'{a[4:]} predice {b[4:]}'] = {'modelo': met(te['pm10'], predecir(entrenar(tr), te)), 'linea_base': met(te['pm10'], base(tr, te))}

imp = permutation_importance(m_cv, D.loc[tei, VARS], np.log(D.loc[tei, 'pm10']), n_repeats=5, random_state=0)
importancia = {v: round(float(s), 3) for v, s in sorted(zip(VARS, imp.importances_mean), key=lambda x: -x[1])}
for k, v in val.items():
    print(f"{k:28s} modelo R²={v['modelo']['r2']:6.3f}  MAE={v['modelo']['mae']:5.1f}  r={v['modelo']['correlacion']:.2f}   |  línea base R²={v['linea_base']['r2']:6.3f}  MAE={v['linea_base']['mae']:5.1f}")
print('Importancia:', importancia)

dias_no_vistos               modelo R²= 0.415  MAE= 14.6  r=0.65   |  línea base R²= 0.233  MAE= 17.6
promedios_mensuales          modelo R²= 0.581  MAE= 10.1  r=0.78   |  línea base R²= 0.299  MAE= 14.2
futuro_2025_2026             modelo R²=-0.071  MAE= 16.4  r=0.48   |  línea base R²=-0.063  MAE= 19.2
Kanata predice SEMAPA        modelo R²=-0.930  MAE= 17.1  r=0.63   |  línea base R²=-1.037  MAE= 19.8
SEMAPA predice Kanata        modelo R²= 0.157  MAE= 21.3  r=0.70   |  línea base R²= 0.040  MAE= 23.1
Importancia: {'estacion_cos': 0.835, 'lst': 0.117, 'focos_7d': 0.1, 't2m': 0.042, 'aod_7d': 0.038, 'humedad': 0.037, 'lluvia': 0.019, 'dsem': 0.017, 'lluvia_3d': 0.016, 'viento': 0.011, 'aod': 0.009, 'u10': 0.009, 'v10': 0.003, 'capa_mezcla': 0.001, 'estacion_sen': 0.0, 'focos': -0.009, 'dias_secos': -0.02}


## Celda B5 · Sensado virtual en los 15 distritos y exportación

Al terminar, descarga `sensado_virtual.json` a `public/data/` y guarda el cuaderno como `notebooks/modelo_sensado.ipynb`.

In [13]:
final = entrenar(D)
q10, q90 = entrenar(D, 0.1), entrenar(D, 0.9)
Z = X[X['zona'].str.startswith('d_')].copy()
Z['pm10'], Z['p10'], Z['p90'] = predecir(final, Z), predecir(q10, Z), predecir(q90, Z)

ref = D[VARS_FISICAS].agg(['mean', 'std'])
def confianza(g):
    z = ((g[VARS_FISICAS].mean() - ref.loc['mean']) / ref.loc['std']).abs().mean()
    return 'alta' if z < 0.5 else 'media' if z < 1 else 'baja'

def analizar(serie):
    s = serie.set_index('fecha').asfreq('MS'); v = s.iloc[:, 0].astype(float)
    r = mk.seasonal_test(v.values, period=12)
    return {'sen_pendiente_anual': round(float(r.slope), 3), 'mk_p': round(float(r.p), 4),
            'mk_tendencia': r.trend, 'significativa': bool(r.p < 0.05)}

distritos_out = []
for zona, g in Z.groupby('zona'):
    mm = g.set_index('fecha')[['pm10', 'p10', 'p90']].resample('MS').mean().dropna()
    distritos_out.append({'id': zona[2:], 'nombre': NOMBRES[zona], 'confianza': confianza(g),
        'pm10_promedio': round(float(g['pm10'].mean()), 1),
        'dias_sobre_oms_pct': round(float((g['pm10'] > 45).mean() * 100), 1),
        'serie': [{'fecha': f.strftime('%Y-%m'), 'pm10': round(float(r.pm10), 1), 'p10': round(float(r.p10), 1),
                   'p90': round(float(r.p90), 1)} for f, r in mm.iterrows()],
        'tendencia': analizar(mm[['pm10']].rename_axis('fecha').reset_index())})

salida = {
    'meta': {'objetivo': 'PM10 diario (µg/m³)', 'modelo': 'Gradient Boosting (scikit-learn HistGradientBoostingRegressor), entrenado en escala logarítmica',
             'variables': VARS, 'periodo': f'{INICIO} a {FIN}', 'validacion': val, 'importancia': importancia,
             'estaciones': {n: {'lon': xy[0], 'lat': xy[1]} for n, xy in ESTACIONES.items()},
             'fuentes': ['NASA MODIS MCD19A2 (AOD)', 'NASA MODIS MOD11A1 (LST)', 'NASA MERRA-2 (meteorología)',
                         'NASA GPM IMERG V07 (lluvia)', 'NASA FIRMS (focos de incendio)', 'Red MÓNICA GAMC (PM10)'],
             'como_leer': {'dias_no_vistos': 'Precisión en meses que el modelo no usó para aprender (la validación principal).',
                           'promedios_mensuales': 'La misma prueba, comparando promedios de cada mes, que es lo que muestra la app.',
                           'futuro_2025_2026': 'Entrenado hasta 2024 y probado en 2025-2026. Le cuesta porque el PM10 bajó mucho en esos años.',
                           'entre_estaciones': 'Entrenado con una estación y probado en la otra. Sigue bien las subidas y bajadas (correlación), pero no el nivel exacto de cada lugar.'},
             'nota': 'Con solo 2 estaciones de referencia, el modelo sigue bien cuándo el aire empeora o mejora, pero el nivel exacto en distritos sin sensor es aproximado (ver campo confianza).'},
    'estaciones_prueba': [{'zona': z[4:], 'serie': [{'fecha': str(p), 'observado': round(float(r.pm10), 1), 'estimado': round(float(r.estimado), 1)}
                                                    for p, r in g.droplevel(0).iterrows()]} for z, g in MM.groupby(level=0)],
    'distritos': sorted(distritos_out, key=lambda d: int(d['id']))}
json.dump(salida, open('sensado_virtual.json', 'w', encoding='utf-8'), ensure_ascii=False)
print(pd.DataFrame([{k: d[k] for k in ['nombre', 'pm10_promedio', 'dias_sobre_oms_pct', 'confianza']} | d['tendencia'] for d in salida['distritos']]))

         nombre  pm10_promedio  dias_sobre_oms_pct confianza  \
0    Distrito 1           61.9                83.1      alta   
1    Distrito 2           56.3                66.3      alta   
2    Distrito 3           57.5                68.0      alta   
3    Distrito 4           60.0                69.2      alta   
4    Distrito 5           61.2                69.2      alta   
5    Distrito 6           61.1                72.8      alta   
6    Distrito 7           67.0                88.6     media   
7    Distrito 8           66.6                88.1     media   
8    Distrito 9           60.6                70.0      alta   
9   Distrito 10           58.9                68.3      alta   
10  Distrito 11           57.9                66.7      alta   
11  Distrito 12           57.2                66.9      alta   
12  Distrito 13           54.4                70.7      alta   
13  Distrito 14           67.2                89.0     media   
14  Distrito 15           64.5          